In [ ]:
from pathlib import Path
import base64
import gzip
import hashlib
import io
import tarfile
import zlib

EXPECTED_SOURCE_SHA = "c6f96a8521dc9aa369b6f27e5b36b9d481e5c1688f50c8ca215c3bb53f1f9eb8"
EXPECTED_ARCHIVE_SHA = "d65410a4fc8a31226055e43085131468ac8af5721aabdf3593cb8eade59cabe3"
PAYLOAD = "..."

source = zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(source).hexdigest() == EXPECTED_SOURCE_SHA
compile(source, "main.py", "exec")
Path("main.py").write_bytes(source)

tar_buffer = io.BytesIO()
with tarfile.open(fileobj=tar_buffer, mode="w") as archive:
    info = tarfile.TarInfo("main.py")
    info.size = len(source)
    info.mode = 0o644
    info.mtime = 0
    info.uid = 0
    info.gid = 0
    info.uname = ""
    info.gname = ""
    archive.addfile(info, io.BytesIO(source))

with Path("submission.tar.gz").open("wb") as raw:
    with gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as compressed:
        compressed.write(tar_buffer.getvalue())

archive_sha = hashlib.sha256(Path("submission.tar.gz").read_bytes()).hexdigest()
assert archive_sha == EXPECTED_ARCHIVE_SHA
with tarfile.open("submission.tar.gz", "r:gz") as archive:
    assert archive.getnames() == ["main.py"]
    assert archive.extractfile("main.py").read() == source

print({
    "agent": "V21-R1",
    "main_py_sha256": EXPECTED_SOURCE_SHA,
    "archive_sha256": archive_sha,
    "archive_members": ["main.py"],
})



# V21-R1 (2300) + Teacher-Anchored Counterfactual DAgger

**The V21-R1 agent is the frozen teacher and remains executable inside the final policy.**
There is no lossy behavioral-cloning replacement of farmer/hand actions.

The learned part is a sparse residual policy over **market SELL decisions only**:

- farmer and every hand action are always copied from the exact teacher;
- HIRE / BUY_LAND / BUY_SEED / BUY_ANIMAL / BUY_PRODUCT remain teacher-exact;
- a residual candidate is trained only from paired full-season counterfactuals with the
  same seed, opponent, seat and identical pre-decision observation hash;
- W/L/T at the end of the 720-turn season is the primary label; money margin is only a
  lower-weight tie-breaker;
- DAgger re-collects labels on states induced by the current residual student;
- calibration uses disjoint seeds;
- an untouched full-game holdout is the only stage allowed to promote the student;
- if promotion fails, `submission.tar.gz` is the exact original V21-R1 teacher.

The candidate set is deliberately constrained: alternative SELL schedules from the
embedded Moon / Mutoy / Munib experts plus one-step hold / premium-hold / half-sell /
slight front-load residuals. Terminal liquidation (steps 708+) is always teacher-exact.


In [ ]:

from pathlib import Path
import copy
import hashlib
import importlib
import json
import os
import random
import shutil
import sys
import tarfile
import tempfile
import time
import types
import urllib.request
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
TEACHER_PATH = WORK / "teacher_v21_r1_main.py"
TEACHER_SOURCE = Path("main.py").read_text(encoding="utf-8")
TEACHER_SHA = hashlib.sha256(TEACHER_SOURCE.encode()).hexdigest()
assert TEACHER_SHA == EXPECTED_SOURCE_SHA, (TEACHER_SHA, EXPECTED_SOURCE_SHA)
TEACHER_PATH.write_text(TEACHER_SOURCE, encoding="utf-8")

# Immutable ready-to-submit artifact before any training.
_current_archive = Path("submission.tar.gz")
assert _current_archive.exists()
shutil.copy2(_current_archive, WORK / "submission_teacher_exact.tar.gz")
if _current_archive.resolve() != (WORK / "submission.tar.gz").resolve():
    shutil.copy2(_current_archive, WORK / "submission.tar.gz")
(WORK / "teacher_sha256.txt").write_text(TEACHER_SHA + "\n", encoding="utf-8")

# Intentionally expensive. The strong 2300 policy is the behavior prior; training is
# allowed to spend hours on exact full-season counterfactuals.
DEEP_MODE = True
MAX_WORKERS = max(1, min(8, (os.cpu_count() or 2)))

DISCOVERY_SEEDS = list(range(410_000, 410_064 if DEEP_MODE else 410_016))
DAGGER1_SEEDS   = list(range(420_000, 420_032 if DEEP_MODE else 420_008))
DAGGER2_SEEDS   = list(range(425_000, 425_024 if DEEP_MODE else 425_006))
CAL_SEEDS       = list(range(430_000, 430_032 if DEEP_MODE else 430_008))
HOLDOUT_SEEDS   = list(range(440_000, 440_048 if DEEP_MODE else 440_012))
DIRECT_SEEDS    = list(range(450_000, 450_032 if DEEP_MODE else 450_008))

OPPONENTS_PER_SEED = 2 if DEEP_MODE else 1
BRANCHES_R0 = 12 if DEEP_MODE else 5
BRANCHES_R1 = 9 if DEEP_MODE else 4
BRANCHES_R2 = 8 if DEEP_MODE else 3
BRANCHES_CAL = 9 if DEEP_MODE else 4

N_TREES = 128 if DEEP_MODE else 64
TREE_DEPTH = 8
MIN_LEAF = 10
MARGIN_EPS = 750.0
MAX_OVERRIDES = 24
EXTERNAL_OPPONENTS = True

ENGINE_COMMIT = "28b6d8af3ce73926b3d0fda1410c1ddd8384ab8c"
ENGINE_PATCH_FILE = WORK / "kaggriculture_offline_patch.py"
VARIANT_DIR = WORK / "teacher_residual_variants"
VARIANT_DIR.mkdir(parents=True, exist_ok=True)

print({
    "teacher_sha256": TEACHER_SHA,
    "exact_fallback": str(WORK / "submission_teacher_exact.tar.gz"),
    "deep_mode": DEEP_MODE,
    "workers": MAX_WORKERS,
    "r0_seeds": len(DISCOVERY_SEEDS),
    "dagger1_seeds": len(DAGGER1_SEEDS),
    "dagger2_seeds": len(DAGGER2_SEEDS),
    "cal_seeds": len(CAL_SEEDS),
    "holdout_seeds": len(HOLDOUT_SEEDS),
    "n_trees": N_TREES,
    "external_opponents": EXTERNAL_OPPONENTS,
})


In [ ]:
# Network-free current-engine bootstrap.
# Kaggle Internet can be disabled, so patch installed >=1.32.5 in memory.

ENGINE_PATCH_SOURCE = '# Offline Kaggriculture engine patch for Kaggle notebook rollouts.\n# Target semantics: Kaggle/kaggle-environments commit\n# 28b6d8af3ce73926b3d0fda1410c1ddd8384ab8c (Aug 15, 2026).\n\nimport inspect\nimport importlib.metadata\nimport textwrap\n\nTARGET_COMMIT = "28b6d8af3ce73926b3d0fda1410c1ddd8384ab8c"\n\n\ndef _pkg_version():\n    for name in ("kaggle-environments", "kaggle_environments"):\n        try:\n            return importlib.metadata.version(name)\n        except Exception:\n            pass\n    return "unknown"\n\n\ndef _replace_old_shop_unlock(kg):\n    # Keep the rest of _end_of_day unchanged; replace only unique-shop selection.\n    if getattr(kg, "_IL_AUG7_SHOP_PATCH_DONE", False):\n        return False\n    src = textwrap.dedent(inspect.getsource(kg._end_of_day))\n    anchor = \'remaining = [s for s in SHOPS if s not in town["unlocked_shops"]]\'\n    if anchor not in src:\n        kg._IL_AUG7_SHOP_PATCH_DONE = True\n        return False\n    old = (\n        \'        remaining = [s for s in SHOPS if s not in town["unlocked_shops"]]\\n\'\n        \'        if remaining:\\n\'\n        \'            choice = rng.choice(sorted(remaining))\\n\'\n        \'            town["unlocked_shops"].append(choice)\'\n    )\n    new = (\n        \'        if len(town["unlocked_shops"]) < MAX_SHOP_INSTANCES:\\n\'\n        \'            town["unlocked_shops"].append(rng.choice(sorted(SHOPS)))\'\n    )\n    if old not in src:\n        raise RuntimeError("old _end_of_day shop block found but exact patch anchor did not match")\n    patched = src.replace(old, new, 1)\n    ns = kg.__dict__\n    exec(compile(patched, "<kaggriculture_end_of_day_aug7_patch>", "exec"), ns, ns)\n    kg._IL_AUG7_SHOP_PATCH_DONE = True\n    return True\n\n\ndef activate_engine():\n    import kaggle_environments as ke\n    from kaggle_environments import make as real_make\n    from kaggle_environments.envs.kaggriculture import kaggriculture as kg\n\n    base_version = _pkg_version()\n    try:\n        nums = tuple(int(x) for x in base_version.split(".")[:3])\n    except Exception:\n        nums = ()\n    if nums and nums < (1, 32, 5):\n        raise RuntimeError(\n            f"kaggle-environments {base_version} is too old for safe offline patching; need >=1.32.5"\n        )\n\n    # Aug-7 town rebalance.\n    kg.MAX_SHOP_INSTANCES = 8\n\n    def town_consume(env, state, step):\n        obs0 = state[0].observation\n        market = obs0.market\n        town = obs0.town\n        cfg = env.configuration\n        shop_interval = max(1, int(kg.get(cfg, "townShopSellInterval", 4)))\n        center_interval = max(1, int(kg.get(cfg, "townCenterSellInterval", 24)))\n        if step % shop_interval == 0:\n            for shop_name in town.get("unlocked_shops", []):\n                products = kg.SHOPS[shop_name]\n                multiplier = 2 if len(products) == 1 else 1\n                for item in products:\n                    market["inventory"][item] -= multiplier\n        if step % center_interval == 0:\n            for item in kg.TOWN_CENTER_PRODUCTS:\n                market["inventory"][item] -= 1\n        kg._refresh_prices(market)\n\n    kg._town_consume = town_consume\n    shop_unlock_patched = _replace_old_shop_unlock(kg)\n\n    # Aug-15 hinge scarcity pricing.\n    kg.HINGE_GAIN = 8.0\n    kg.MARKET_PARAMS["CARROT"].update(below_func="hinge", below_target=1.00)\n    kg.MARKET_PARAMS["TOMATO"].update(below_func="hinge", below_target=0.40)\n    kg.MARKET_PARAMS["EGG"].update(below_func="hinge", below_target=0.40)\n\n    def shape(func, x, T=None):\n        x = max(0.0, x)\n        if func == "linear": return x\n        if func == "sq": return x * x\n        if func == "sqrt": return kg.math.sqrt(x)\n        if func == "log": return kg.math.log(1.0 + x)\n        if func == "log10": return kg.math.log10(1.0 + x)\n        if func == "hinge":\n            if not T or T <= 0:\n                return x\n            u = x / T\n            return u + kg.HINGE_GAIN * max(0.0, u - 1.0) ** 2\n        return x\n\n    def market_price(item, inventory, params=None):\n        p = (params or kg.MARKET_PARAMS)[item]\n        base = p["base"]\n        I0 = p["I0"]\n        T = p["T"]\n        if inventory < I0:\n            f = p["below_func"]\n            amp = p["below_target"] * base / shape(f, T, T)\n            price = base + amp * shape(f, I0 - inventory, T)\n        else:\n            f = p["above_func"]\n            amp = p["above_target"] * base / shape(f, T, T)\n            price = base - amp * shape(f, inventory - I0, T)\n        return max(kg.PRICE_FLOOR, int(round(price)))\n\n    kg._shape = shape\n    kg.market_price = market_price\n\n    # The 1.32.5 JSON spec has center interval default 12; override it to 24.\n    def patched_make(name, *args, configuration=None, **kwargs):\n        if name == "kaggriculture":\n            cfg = dict(configuration or {})\n            cfg.setdefault("townCenterSellInterval", 24)\n            configuration = cfg\n        return real_make(name, *args, configuration=configuration, **kwargs)\n\n    report = {\n        "base_package_version": base_version,\n        "target_commit": TARGET_COMMIT,\n        "shop_unlock_was_patched": bool(shop_unlock_patched),\n        "carrot_func": kg.MARKET_PARAMS["CARROT"]["below_func"],\n        "tomato_func": kg.MARKET_PARAMS["TOMATO"]["below_func"],\n        "egg_func": kg.MARKET_PARAMS["EGG"]["below_func"],\n        "carrot_9400": kg.market_price("CARROT", 9400),\n        "tomato_9700": kg.market_price("TOMATO", 9700),\n        "egg_9502": kg.market_price("EGG", 9502),\n        "max_shop_instances": getattr(kg, "MAX_SHOP_INSTANCES", None),\n    }\n    return ke, patched_make, kg, report\n'
ENGINE_PATCH_FILE.write_text(ENGINE_PATCH_SOURCE, encoding="utf-8")
compile(ENGINE_PATCH_SOURCE, str(ENGINE_PATCH_FILE), "exec")

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
from kaggriculture_offline_patch import activate_engine

_ke, make, kg, _engine_report = activate_engine()
_expected_contract = {
    "carrot_func": "hinge",
    "tomato_func": "hinge",
    "egg_func": "hinge",
    "carrot_9400": 113,
    "tomato_9700": 144,
    "egg_9502": 120,
    "max_shop_instances": 8,
}
for key, expected in _expected_contract.items():
    actual = _engine_report[key]
    if actual != expected:
        raise RuntimeError(f"offline engine contract mismatch: {key}={actual!r}, expected {expected!r}")

_probe = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 123456}, debug=False)
assert int(_probe.configuration.episodeSteps) == 720
assert int(_probe.configuration.townCenterSellInterval) == 24
print("offline engine patch OK")
print("engine report:", _engine_report)
print("townCenterSellInterval:", int(_probe.configuration.townCenterSellInterval))
print("network required: False")


In [ ]:

import base64 as _b64
import zlib as _zlib

POP_DIR = WORK / "opponent_population"
POP_DIR.mkdir(parents=True, exist_ok=True)

def load_source_module(source, name):
    mod = types.ModuleType(name)
    mod.__file__ = f"<{name}>"
    exec(compile(source, mod.__file__, "exec"), mod.__dict__)
    return mod

_teacher_mod = load_source_module(TEACHER_SOURCE, "_teacher_population_probe")

def _decode_payload(text):
    return _zlib.decompress(_b64.b85decode(text)).decode("utf-8")

MOON_SOURCE = _decode_payload(_teacher_mod._MOON_PAYLOAD)
MUTOY_SOURCE = _decode_payload(_teacher_mod._MUTOY_PAYLOAD)
MUNIB_SOURCE = _decode_payload(_teacher_mod._MUNIB_PAYLOAD)
MUNIB_FRONT_SOURCE = MUNIB_SOURCE.rstrip() + '\n\n_FR_ITEMS = ("WOOL", "MILK", "MELON", "STRAWBERRY")\n'

internal_sources = {
    "teacher_v21_r1": TEACHER_SOURCE,
    "moon": MOON_SOURCE,
    "mutoy": MUTOY_SOURCE,
    "munib_base": MUNIB_SOURCE,
    "munib_front": MUNIB_FRONT_SOURCE,
}
population_paths = {"starter": "builtin:starter"}
for name, source in internal_sources.items():
    compile(source, f"{name}.py", "exec")
    p = POP_DIR / f"{name}.py"
    p.write_text(source, encoding="utf-8")
    population_paths[name] = str(p)

# Optional public diversity. Training remains valid if Kaggle Internet is disabled.
external_urls = {
    "gzmcr_current": "https://raw.githubusercontent.com/GzmCR/Kaggriculture/main/main.py",
    "cok_current": "https://raw.githubusercontent.com/COK-ZhangZiliang/Kaggriculture/main/main.py",
    "lonespear_current": "https://raw.githubusercontent.com/lonespear/kaggriculture/main/main.py",
    "seyamalam_current": "https://raw.githubusercontent.com/Seyamalam/Kaggriculture/main/main.py",
}
external_errors = {}
if EXTERNAL_OPPONENTS:
    for name, url in external_urls.items():
        try:
            with urllib.request.urlopen(url, timeout=20) as r:
                source = r.read().decode("utf-8")
            compile(source, f"{name}.py", "exec")
            p = POP_DIR / f"{name}.py"
            p.write_text(source, encoding="utf-8")
            population_paths[name] = str(p)
        except Exception as exc:
            external_errors[name] = repr(exc)

assert "teacher_v21_r1" in population_paths
assert len(population_paths) >= 6
print("opponents:", sorted(population_paths))
if external_errors:
    print("external opponents skipped:", external_errors)



## Why this is actual teacher-anchored imitation

The previous three-gate router had almost no learnable action space. Here the teacher is
the **behavior prior**, not a dataset that gets approximated and thrown away.

For a visited state \(s_t\), training compares:

\[
Q(s_t, a_{\text{teacher}}) \quad \text{vs} \quad Q(s_t, a_{\text{residual}})
\]

by replaying the full season twice with the same prefix. The student learns the binary
advantage decision **"override the teacher here or abstain"**. When uncertain it executes
the teacher action exactly. This is advantage-weighted residual imitation with DAgger,
not raw action cloning.


In [ ]:
IL_OVERLAY_TEMPLATE = r'''
# === TEACHER-ANCHORED COUNTERFACTUAL DAGGER RESIDUAL ===
# Frozen V21-R1 remains the executable policy prior.
# Student may change only SELL timing/quantities; farmer/hands and all capital orders stay teacher-exact.

import math as _il_math
import json as _il_json
import hashlib as _il_hashlib
import copy as _il_copy
import types as _il_types

_IL_MODEL = __MODEL_BUNDLE__
_IL_ENABLE_LOG = __ENABLE_LOG__
_IL_FORCE_STEP = None
_IL_FORCE_CANDIDATE = None
_IL_PRODUCTS = ("WHEAT","CARROT","TOMATO","STRAWBERRY","MELON","EGG","MILK","WOOL","FERTILIZER")
_IL_SHOPS = ("BAKERY","PIZZA_SHOP","BRUNCH_SPOT","YARN_STORE","ICE_CREAM_SHOP","PET_CAFE","SMOOTHIE_SHOP","FARMERS_MARKET")
_IL_BASE_PRICE = {"WHEAT":25.0,"CARROT":35.0,"TOMATO":60.0,"STRAWBERRY":120.0,"MELON":250.0,"EGG":50.0,"MILK":160.0,"WOOL":200.0,"FERTILIZER":100.0}
_IL_SCALE_T = {"WHEAT":400.0,"CARROT":450.0,"TOMATO":200.0,"STRAWBERRY":100.0,"MELON":300.0,"EGG":332.0,"MILK":122.0,"WOOL":105.0,"FERTILIZER":200.0}
_IL_CANDIDATE_NAMES = {0:"teacher",1:"moon",2:"munib_base",3:"munib_front",4:"mutoy",5:"hold_all",6:"hold_premium",7:"half_sells",8:"boost_sells"}
_IL_PREMIUM = ("STRAWBERRY","MELON","MILK","WOOL")
_IL_LOG = {0: [], 1: []}
_IL_TARGET_LOG = {0: [], 1: []}
_IL_OVERRIDE_COUNT = {0: 0, 1: 0}
_IL_LAST_STEP = {0: -1, 1: -1}

# Freeze the exact original function before replacing the public entrypoint.
_IL_TEACHER = agent

# Shadow experts are separate modules: probing them cannot mutate teacher expert state.
_IL_SHADOW_MOON = _load_payload(_MOON_PAYLOAD, "_il_shadow_moon")
_IL_SHADOW_MUTOY = _load_payload(_MUTOY_PAYLOAD, "_il_shadow_mutoy")
_IL_SHADOW_MUNIB_BASE = _load_payload(_MUNIB_PAYLOAD, "_il_shadow_munib_base")
_IL_SHADOW_MUNIB_FRONT = _load_payload(_MUNIB_PAYLOAD, "_il_shadow_munib_front")
_IL_SHADOW_MUNIB_FRONT._FR_ITEMS = ("WOOL", "MILK", "MELON", "STRAWBERRY")

def _il_call(module, obs, configuration=None):
    entry = getattr(module, "_kaggle_submission_entrypoint", getattr(module, "agent"))
    isolated = _il_copy.deepcopy(obs)
    try:
        return entry(isolated, configuration)
    except TypeError:
        return entry(isolated)

def _il_norm_action(action, obs):
    seat = _seat(obs)
    farms = list(obs.get("farms", []) or [])
    own = farms[seat] if len(farms) > seat else {}
    n_hands = len(list(own.get("hands", []) or []))
    a = _il_copy.deepcopy(action or {})
    farmer = a.get("farmer")
    if not isinstance(farmer, list) or not farmer:
        farmer = ["PASS"]
    hands = list(a.get("hands", []) or [])
    hands = [(h if isinstance(h,list) and h else ["PASS"]) for h in hands[:n_hands]]
    while len(hands) < n_hands:
        hands.append(["PASS"])
    market = [list(x) for x in list(a.get("market", []) or []) if isinstance(x,(list,tuple)) and x]
    return {"farmer": list(farmer), "hands": hands, "market": market[:10]}

def _il_actor_sig(action):
    return (
        tuple((action or {}).get("farmer", []) or []),
        tuple(tuple(x) for x in ((action or {}).get("hands", []) or [])),
    )

def _il_market_sig(action):
    return tuple(tuple(x) for x in ((action or {}).get("market", []) or []))

def _il_sells(action):
    out = []
    for order in list((action or {}).get("market", []) or []):
        if isinstance(order, (list,tuple)) and len(order) >= 3 and str(order[0]) == "SELL":
            try:
                q = max(0, int(order[2] or 0))
            except Exception:
                q = 0
            if q:
                out.append(["SELL", str(order[1]), q])
    return out

def _il_non_sells(action):
    out = []
    for order in list((action or {}).get("market", []) or []):
        if not (isinstance(order,(list,tuple)) and len(order) >= 1 and str(order[0]) == "SELL"):
            out.append(list(order))
    return out

def _il_merge_sells(teacher_action, sell_source):
    out = _il_copy.deepcopy(teacher_action)
    market = _il_non_sells(teacher_action)
    # Aggregate duplicate SELLs from the proposal so candidate identity is stable.
    totals = {}
    order_items = []
    for order in _il_sells(sell_source):
        item, q = order[1], int(order[2])
        if item not in totals:
            order_items.append(item)
            totals[item] = 0
        totals[item] += q
    for item in order_items:
        if len(market) >= 10:
            break
        q = int(totals[item])
        if q > 0:
            market.append(["SELL", item, q])
    out["market"] = market[:10]
    return out

def _il_hold_candidate(teacher_action, items=None):
    out = _il_copy.deepcopy(teacher_action)
    market = _il_non_sells(teacher_action)
    blocked = None if items is None else set(items)
    for order in _il_sells(teacher_action):
        if blocked is None or order[1] in blocked:
            continue
        if len(market) < 10:
            market.append(list(order))
    out["market"] = market[:10]
    return out

def _il_scale_sells(teacher_action, factor):
    out = _il_copy.deepcopy(teacher_action)
    market = _il_non_sells(teacher_action)
    for order in _il_sells(teacher_action):
        if len(market) >= 10:
            break
        q = max(0, int(round(float(order[2]) * float(factor))))
        if q > 0:
            market.append(["SELL", order[1], q])
    out["market"] = market[:10]
    return out

def _il_state_hash(obs):
    payload = {
        "step": obs.get("step"), "day": obs.get("day"), "hour": obs.get("hour"),
        "player": obs.get("player"), "farms": obs.get("farms"),
        "private": obs.get("private"), "market": obs.get("market"), "town": obs.get("town"),
    }
    raw = _il_json.dumps(payload, sort_keys=True, separators=(",",":"), default=str)
    return _il_hashlib.sha256(raw.encode("utf-8")).hexdigest()

def _il_clip(x, lo=-6.0, hi=6.0):
    try:
        x = float(x)
    except Exception:
        return 0.0
    return lo if x < lo else hi if x > hi else x

def _il_logmoney(x):
    try:
        return _il_math.log1p(max(0.0,float(x))) / 13.0
    except Exception:
        return 0.0

def _il_sell_totals(action):
    d = {p:0.0 for p in _IL_PRODUCTS}
    for order in _il_sells(action):
        if order[1] in d:
            d[order[1]] += float(order[2])
    return d

def _il_candidate_features(obs, teacher_action, candidate_action, candidate_id):
    seat = _seat(obs)
    step = int(obs.get("step",0) or 0)
    day = int(obs.get("day",step//24) or 0)
    hour = int(obs.get("hour",step%24) or 0)
    farms = list(obs.get("farms",[]) or [])
    own = farms[seat] if len(farms)>seat else {}
    opp = farms[1-seat] if len(farms)>=2 else {}
    private = obs.get("private",{}) or {}
    shed = private.get("shed",{}) or {}
    inventories = list(private.get("inventories",[]) or [])
    market = obs.get("market",{}) or {}
    prices = market.get("prices",{}) or {}
    inv = market.get("inventory",{}) or {}
    shops = list((obs.get("town",{}) or {}).get("unlocked_shops",[]) or [])

    f = [
        float(seat),
        step/719.0, day/29.0, hour/23.0,
        _il_logmoney(own.get("money",0)), _il_logmoney(opp.get("money",0)),
        _il_clip((float(own.get("money",0) or 0)-float(opp.get("money",0) or 0))/50000.0),
        min(1.0,len(list(own.get("hands",[]) or []))/20.0),
        min(1.0,len(list(opp.get("hands",[]) or []))/20.0),
        min(1.0,float(own.get("hires_today",0) or 0)/20.0),
        min(1.0,float(opp.get("hires_today",0) or 0)/20.0),
        min(1.0,len(list(own.get("unlocked_quadrants",[]) or []))/4.0),
        min(1.0,len(list(opp.get("unlocked_quadrants",[]) or []))/4.0),
        min(1.0,len(shops)/8.0),
    ]
    counts = {s:0 for s in _IL_SHOPS}
    for s in shops:
        if s in counts:
            counts[s] += 1
    for s in _IL_SHOPS:
        f.append(counts[s]/8.0)

    carry = {p:0.0 for p in _IL_PRODUCTS}
    for bag in inventories:
        if isinstance(bag,dict):
            for p in _IL_PRODUCTS:
                carry[p] += max(0.0,float(bag.get(p,0) or 0))

    ts = _il_sell_totals(teacher_action)
    cs = _il_sell_totals(candidate_action)
    teacher_value = 0.0
    cand_value = 0.0
    for p in _IL_PRODUCTS:
        base = _IL_BASE_PRICE[p]
        qprice = float(prices.get(p,base) or 0)
        qinv = float(inv.get(p,10000) or 0)
        teacher_value += ts[p]*qprice
        cand_value += cs[p]*qprice
        f.extend([
            _il_clip(qprice/base,0.0,8.0)/8.0,
            _il_clip((10000.0-qinv)/_IL_SCALE_T[p],-5.0,5.0)/5.0,
            min(2.0,max(0.0,float(shed.get(p,0) or 0))/100.0),
            min(2.0,carry[p]/100.0),
            min(2.0,ts[p]/100.0),
            min(2.0,cs[p]/100.0),
            _il_clip((cs[p]-ts[p])/100.0,-2.0,2.0)/2.0,
        ])
    f.extend([
        _il_clip(teacher_value/50000.0,-3.0,3.0)/3.0,
        _il_clip(cand_value/50000.0,-3.0,3.0)/3.0,
        _il_clip((cand_value-teacher_value)/50000.0,-3.0,3.0)/3.0,
        min(1.0,len(_il_sells(teacher_action))/10.0),
        min(1.0,len(_il_sells(candidate_action))/10.0),
    ])
    route = str((_ROUTE.get(seat) if isinstance(_ROUTE,dict) else None) or "")
    f.extend([
        1.0 if route == "moon" else 0.0,
        1.0 if route == "munib" else 0.0,
        1.0 if route == "mutoy" else 0.0,
        1.0 if not route else 0.0,
        1.0 if bool(_MARKET_OVERLAY.get(seat,False)) else 0.0,
    ])
    for cid in range(1,9):
        f.append(1.0 if int(candidate_id)==cid else 0.0)
    return f

def _il_tree_prob(tree, x):
    left, right = tree["left"], tree["right"]
    feature, threshold, prob = tree["feature"], tree["threshold"], tree["prob"]
    node = 0
    while left[node] != -1:
        j = feature[node]
        node = left[node] if float(x[j]) <= threshold[node] else right[node]
    return prob[node]

def _il_predict(features):
    trees = (_IL_MODEL or {}).get("trees") or []
    if not trees:
        return None, None
    ps = [_il_tree_prob(t, features) for t in trees]
    p = sum(ps)/len(ps)
    var = sum((q-p)*(q-p) for q in ps)/len(ps)
    return p, var**0.5

def _il_candidates(obs, teacher_action, configuration=None):
    # Shadow calls are state-isolated from the frozen teacher.
    shadow = {
        1: _il_norm_action(_il_call(_IL_SHADOW_MOON, obs, configuration), obs),
        2: _il_norm_action(_il_call(_IL_SHADOW_MUNIB_BASE, obs, configuration), obs),
        3: _il_norm_action(_il_call(_IL_SHADOW_MUNIB_FRONT, obs, configuration), obs),
        4: _il_norm_action(_il_call(_IL_SHADOW_MUTOY, obs, configuration), obs),
    }
    out = {}
    seen = set()
    teacher_sig = _il_actor_sig(teacher_action)
    teacher_market = _il_market_sig(teacher_action)
    seen.add(teacher_market)
    for cid, proposal in shadow.items():
        # Only borrow SELL timing from an expert whose field action agrees exactly
        # with the teacher at this state.
        if _il_actor_sig(proposal) != teacher_sig:
            continue
        merged = _il_merge_sells(teacher_action, proposal)
        sig = _il_market_sig(merged)
        if sig not in seen:
            out[cid] = merged
            seen.add(sig)

    synthetic = {
        5: _il_hold_candidate(teacher_action, None),
        6: _il_hold_candidate(teacher_action, _IL_PREMIUM),
        7: _il_scale_sells(teacher_action, 0.50),
        8: _il_scale_sells(teacher_action, 1.25),
    }
    for cid, cand in synthetic.items():
        sig = _il_market_sig(cand)
        if sig not in seen:
            out[cid] = cand
            seen.add(sig)
    return out

def _il_priority(obs, teacher_action, candidate_action):
    prices = ((obs.get("market") or {}).get("prices") or {})
    ts, cs = _il_sell_totals(teacher_action), _il_sell_totals(candidate_action)
    value = 0.0
    volume = 0.0
    for p in _IL_PRODUCTS:
        dq = cs[p]-ts[p]
        value += abs(dq)*float(prices.get(p,_IL_BASE_PRICE[p]) or 0)
        volume += abs(dq)
    return float(value + 50.0*volume)

def agent(obs, configuration=None):
    seat = _seat(obs)
    step = int(obs.get("step",0) or 0)
    if step == 0 or step < int(_IL_LAST_STEP.get(seat,-1)):
        _IL_LOG[seat] = []
        _IL_TARGET_LOG[seat] = []
        _IL_OVERRIDE_COUNT[seat] = 0
    _IL_LAST_STEP[seat] = step

    teacher_action = _il_norm_action(_IL_TEACHER(_il_copy.deepcopy(obs), configuration), obs)
    candidates = _il_candidates(obs, teacher_action, configuration)

    forced_step = _IL_FORCE_STEP
    forced_cid = _IL_FORCE_CANDIDATE
    selected_id = 0
    selected = teacher_action
    scored = []

    # Never alter the frozen teacher's terminal liquidation.
    can_override = 24 <= step < 708

    if forced_step is not None and int(forced_step) == step:
        cid = int(forced_cid or 0)
        if cid in candidates and can_override:
            selected_id = cid
            selected = candidates[cid]
        else:
            selected_id = 0
            selected = teacher_action
    elif can_override and (_IL_MODEL or {}).get("enabled", False):
        global_threshold = float(_IL_MODEL.get("threshold",0.90))
        global_max_std = float(_IL_MODEL.get("max_std",0.22))
        thresholds = _IL_MODEL.get("thresholds",{}) or {}
        max_stds = _IL_MODEL.get("max_stds",{}) or {}
        min_priority = float(_IL_MODEL.get("min_priority",0.0))
        max_overrides = int(_IL_MODEL.get("max_overrides",24))
        if _IL_OVERRIDE_COUNT.get(seat,0) < max_overrides:
            for cid, cand in candidates.items():
                feat = _il_candidate_features(obs, teacher_action, cand, cid)
                p, std = _il_predict(feat)
                priority = _il_priority(obs, teacher_action, cand)
                scored.append((cid,p,std,priority,feat))
            eligible = []
            for row in scored:
                cid,p,std,priority,feat = row
                threshold = float(thresholds.get(str(cid),global_threshold))
                max_std = float(max_stds.get(str(cid),global_max_std))
                if (
                    p is not None and std is not None
                    and p >= threshold and std <= max_std
                    and priority >= min_priority
                ):
                    eligible.append(row)
            if eligible:
                cid,p,std,priority,feat = max(eligible, key=lambda z:(z[1], z[3], -z[2]))
                selected_id = int(cid)
                selected = candidates[cid]
                _IL_OVERRIDE_COUNT[seat] = int(_IL_OVERRIDE_COUNT.get(seat,0)) + 1

    state_hash = _il_state_hash(obs)
    # Discovery logs one row per valid non-teacher candidate.
    if _IL_ENABLE_LOG:
        known = {row[0]:row for row in scored}
        for cid,cand in candidates.items():
            if cid in known:
                _,p,std,priority,feat = known[cid]
            else:
                feat = _il_candidate_features(obs, teacher_action, cand, cid)
                p,std = _il_predict(feat)
                priority = _il_priority(obs, teacher_action, cand)
            _IL_LOG[seat].append({
                "step":step, "candidate_id":int(cid), "selected_id":int(selected_id),
                "state_hash":state_hash, "features":list(feat),
                "p":p, "std":std, "priority":float(priority),
            })

    if forced_step is not None and int(forced_step) == step:
        cid = int(forced_cid or 0)
        target_feat = None
        if cid in candidates:
            target_feat = _il_candidate_features(obs, teacher_action, candidates[cid], cid)
        _IL_TARGET_LOG[seat].append({
            "step":step, "forced_candidate_id":cid, "applied_candidate_id":int(selected_id),
            "state_hash":state_hash,
            "features": target_feat,
            "available": bool(cid == 0 or cid in candidates),
        })

    return selected

def _kaggle_submission_entrypoint(obs, configuration=None):
    return agent(obs, configuration)
'''

def _pyclean(obj):
    if isinstance(obj, dict):
        return {str(k): _pyclean(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_pyclean(v) for v in obj]
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            pass
    return obj

def build_student_source(model_bundle=None, enable_log=False):
    model_bundle = _pyclean(model_bundle or {})
    text = IL_OVERLAY_TEMPLATE
    text = text.replace("__MODEL_BUNDLE__", repr(model_bundle))
    text = text.replace("__ENABLE_LOG__", repr(bool(enable_log)))
    source = TEACHER_SOURCE.rstrip() + "\n\n" + text
    compile(source, "teacher_residual_student.py", "exec")
    return source

def write_policy(model_bundle=None, name="student.py", enable_log=False):
    source = build_student_source(model_bundle, enable_log=enable_log)
    path = VARIANT_DIR / name
    path.write_text(source, encoding="utf-8")
    return str(path), source

PROBE_PATH, PROBE_SOURCE = write_policy({}, "teacher_probe.py", enable_log=True)
assert PROBE_SOURCE.startswith(TEACHER_SOURCE.rstrip())
assert hashlib.sha256(TEACHER_SOURCE.encode()).hexdigest() == TEACHER_SHA
print({
    "probe": PROBE_PATH,
    "teacher_prefix_bytes": len(TEACHER_SOURCE.encode()),
    "overlay_bytes": len(PROBE_SOURCE.encode()) - len(TEACHER_SOURCE.encode()),
    "status": "compiled",
})


In [ ]:

WORKER_SOURCE = r"""
import random
import sys
import types
import uuid

_WORK_DIR = __WORK_DIR__
if _WORK_DIR not in sys.path:
    sys.path.insert(0, _WORK_DIR)

from kaggriculture_offline_patch import activate_engine
_worker_ke, make, _worker_kg, _worker_engine_report = activate_engine()

_EXPECTED = {
    "carrot_func": "hinge",
    "tomato_func": "hinge",
    "egg_func": "hinge",
    "carrot_9400": 113,
    "tomato_9700": 144,
    "egg_9502": 120,
    "max_shop_instances": 8,
}
for _key, _expected in _EXPECTED.items():
    if _worker_engine_report.get(_key) != _expected:
        raise RuntimeError(f"worker engine mismatch: {_key}={_worker_engine_report.get(_key)!r}")

def _load(path, tag):
    source = open(path, "r", encoding="utf-8").read()
    module = types.ModuleType(f"{tag}_{uuid.uuid4().hex}")
    module.__file__ = path
    exec(compile(source, path, "exec"), module.__dict__)
    return module

def _entry(module):
    return getattr(module, "_kaggle_submission_entrypoint", getattr(module, "agent"))

def run_one(job):
    candidate_path = job["candidate_path"]
    opponent_spec = job["opponent_path"]
    seed = int(job["seed"])
    seat = int(job["seat"])

    random.seed((seed * 1009 + seat * 9176) & 0xFFFFFFFF)
    try:
        import numpy as np
        np.random.seed((seed * 917 + seat * 101) & 0xFFFFFFFF)
    except Exception:
        pass

    try:
        cand = _load(candidate_path, "cand")
        if job.get("force_step") is not None and hasattr(cand, "_IL_FORCE_STEP"):
            cand._IL_FORCE_STEP = int(job["force_step"])
            cand._IL_FORCE_CANDIDATE = int(job.get("force_candidate", 0))

        ce = _entry(cand)
        if isinstance(opponent_spec, str) and opponent_spec.startswith("builtin:"):
            oe = opponent_spec.split(":", 1)[1]
        else:
            opp = _load(opponent_spec, "opp")
            oe = _entry(opp)

        pair = [ce, oe] if seat == 0 else [oe, ce]
        env = make(
            "kaggriculture",
            configuration={"episodeSteps": 720, "seed": seed},
            debug=False,
        )
        env.run(pair)
        final = env.steps[-1]
        c = final[seat]
        o = final[1-seat]
        cr, orr = c.reward, o.reward
        status_c, status_o = str(c.status), str(o.status)

        raw_log = getattr(cand, "_IL_LOG", {})
        raw_target = getattr(cand, "_IL_TARGET_LOG", {})
        log = list(raw_log.get(seat, []) or []) if isinstance(raw_log, dict) else []
        target = list(raw_target.get(seat, []) or []) if isinstance(raw_target, dict) else []
        ok = (
            cr is not None and orr is not None
            and "DONE" in status_c and "DONE" in status_o
        )
        return {
            "ok": bool(ok),
            "candidate_reward": None if cr is None else float(cr),
            "opponent_reward": None if orr is None else float(orr),
            "candidate_status": status_c,
            "opponent_status": status_o,
            "log": log,
            "target_log": target,
            "error": None,
        }
    except Exception as exc:
        return {
            "ok": False,
            "candidate_reward": None,
            "opponent_reward": None,
            "candidate_status": "ERROR",
            "opponent_status": "ERROR",
            "log": [],
            "target_log": [],
            "error": repr(exc),
        }
"""
WORKER_SOURCE = WORKER_SOURCE.replace("__WORK_DIR__", repr(str(WORK.resolve())))
(WORK / "teacher_residual_worker.py").write_text(WORKER_SOURCE, encoding="utf-8")
compile(WORKER_SOURCE, "teacher_residual_worker.py", "exec")

if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
from teacher_residual_worker import run_one as _run_one_worker

def parallel_run(jobs, workers=MAX_WORKERS):
    jobs = list(jobs)
    out = [None] * len(jobs)
    if workers <= 1:
        for i, job in enumerate(jobs):
            out[i] = _run_one_worker(job)
        return out
    with ProcessPoolExecutor(max_workers=workers) as ex:
        future_to_i = {ex.submit(_run_one_worker, job): i for i, job in enumerate(jobs)}
        done = 0
        for fut in as_completed(future_to_i):
            i = future_to_i[fut]
            out[i] = fut.result()
            done += 1
            if done % max(25, len(jobs)//20 or 1) == 0:
                print(f"  completed {done}/{len(jobs)}")
    return out

print("full-season worker ready")


In [ ]:

def _score(result):
    if not result or not result.get("ok"):
        return None, None
    a = float(result["candidate_reward"])
    b = float(result["opponent_reward"])
    score = 1.0 if a > b else 0.5 if a == b else 0.0
    return score, a - b

def _choose_opponents(seed, round_id, count):
    names = sorted(population_paths)
    rng = random.Random(int(seed) * 1_000_003 + int(round_id) * 91_777)
    if count >= len(names):
        return names
    # Force meaningful diversity: teacher/self-play is sampled regularly.
    picked = []
    if "teacher_v21_r1" in names and int(seed) % 3 == 0:
        picked.append("teacher_v21_r1")
    remaining = [n for n in names if n not in picked]
    need = max(0, count - len(picked))
    picked.extend(rng.sample(remaining, min(need, len(remaining))))
    return picked

def run_discovery(policy_path, seeds, round_id, opponents_per_seed=OPPONENTS_PER_SEED):
    jobs, keys = [], []
    for seed in seeds:
        for opp_name in _choose_opponents(seed, round_id, opponents_per_seed):
            opp_path = population_paths[opp_name]
            for seat in (0, 1):
                jobs.append({
                    "candidate_path": policy_path,
                    "opponent_path": opp_path,
                    "seed": int(seed),
                    "seat": int(seat),
                })
                keys.append((int(seed), opp_name, opp_path, int(seat)))
    print(f"discovery r{round_id}: {len(jobs)} games")
    results = parallel_run(jobs)
    worlds = []
    for key, result in zip(keys, results):
        seed, opp_name, opp_path, seat = key
        worlds.append({
            "seed": seed,
            "opponent": opp_name,
            "opponent_path": opp_path,
            "seat": seat,
            "result": result,
        })
    return worlds

def _select_branch_rows(log_rows, budget, seed):
    rows = [
        dict(r) for r in (log_rows or [])
        if 24 <= int(r.get("step", -1)) < 708
        and 1 <= int(r.get("candidate_id", 0)) <= 8
        and r.get("features") is not None
    ]
    # Exact duplicate candidate/state rows are useless.
    unique = {}
    for r in rows:
        key = (int(r["step"]), int(r["candidate_id"]), str(r.get("state_hash","")))
        if key not in unique or float(r.get("priority",0.0)) > float(unique[key].get("priority",0.0)):
            unique[key] = r
    rows = list(unique.values())
    if len(rows) <= budget:
        return rows

    rng = random.Random(int(seed))
    selected, used = [], set()

    # First cover each residual source at least once when available.
    for cid in range(1,9):
        cand = [r for r in rows if int(r["candidate_id"]) == cid]
        if cand:
            best = max(cand, key=lambda r: float(r.get("priority",0.0)))
            k = (int(best["step"]), int(best["candidate_id"]))
            if k not in used and len(selected) < budget:
                selected.append(best); used.add(k)

    # Then cover season phases, preferring economically meaningful disagreements.
    phase_bins = [(24,144),(144,288),(288,432),(432,576),(576,708)]
    for lo, hi in phase_bins:
        cand = [
            r for r in rows
            if lo <= int(r["step"]) < hi
            and (int(r["step"]), int(r["candidate_id"])) not in used
        ]
        if cand and len(selected) < budget:
            cand.sort(key=lambda r: float(r.get("priority",0.0)), reverse=True)
            # Randomize among the top few to avoid always training the same exact turn.
            best = rng.choice(cand[:min(4,len(cand))])
            selected.append(best)
            used.add((int(best["step"]), int(best["candidate_id"])))

    # Fill remaining slots with a 50/50 high-priority + random mix.
    remaining = [
        r for r in rows
        if (int(r["step"]), int(r["candidate_id"])) not in used
    ]
    remaining.sort(key=lambda r: float(r.get("priority",0.0)), reverse=True)
    while remaining and len(selected) < budget:
        if len(selected) % 2 == 0:
            pick_i = 0
        else:
            pick_i = rng.randrange(min(len(remaining), max(1, len(remaining)//2)))
        r = remaining.pop(pick_i)
        k = (int(r["step"]), int(r["candidate_id"]))
        if k in used:
            continue
        selected.append(r); used.add(k)
    return selected[:budget]

def collect_counterfactual_round(
    policy_path,
    seeds,
    round_id,
    branch_budget,
    opponents_per_seed=OPPONENTS_PER_SEED,
):
    worlds = run_discovery(policy_path, seeds, round_id, opponents_per_seed)
    jobs, meta = [], []
    stats = defaultdict(int)

    for wi, world in enumerate(worlds):
        result = world["result"]
        if not result or not result.get("ok"):
            stats["discovery_failure"] += 1
            continue
        rows = _select_branch_rows(
            result.get("log") or [],
            branch_budget,
            seed=world["seed"] * 1009 + world["seat"] * 97 + round_id * 17,
        )
        stats["discovered_candidate_rows"] += len(result.get("log") or [])
        stats["selected_counterfactuals"] += len(rows)
        for ri, row in enumerate(rows):
            base = {
                "candidate_path": policy_path,
                "opponent_path": world["opponent_path"],
                "seed": int(world["seed"]),
                "seat": int(world["seat"]),
                "force_step": int(row["step"]),
            }
            # Pair A: same student prefix, force frozen teacher at target step.
            j0 = dict(base); j0["force_candidate"] = 0
            # Pair B: same prefix, force the proposed residual at target step.
            j1 = dict(base); j1["force_candidate"] = int(row["candidate_id"])
            jobs.extend([j0, j1])
            meta.append({
                "world": world,
                "row": row,
                "pair_index": len(meta),
            })

    print(f"counterfactual r{round_id}: {len(jobs)} branch games ({len(meta)} pairs)")
    results = parallel_run(jobs)

    samples = []
    for k, m in enumerate(meta):
        teacher_res = results[2*k]
        cand_res = results[2*k+1]
        if not teacher_res.get("ok") or not cand_res.get("ok"):
            stats["technical_pair"] += 1
            continue
        tlog = list(teacher_res.get("target_log") or [])
        clog = list(cand_res.get("target_log") or [])
        if len(tlog) != 1 or len(clog) != 1:
            stats["target_log_missing"] += 1
            continue
        tr, cr = tlog[0], clog[0]
        if tr.get("state_hash") != cr.get("state_hash"):
            stats["causal_prefix_mismatch"] += 1
            continue
        if str(tr.get("state_hash")) != str(m["row"].get("state_hash")):
            stats["discovery_prefix_mismatch"] += 1
            continue
        cid = int(m["row"]["candidate_id"])
        if not bool(cr.get("available")) or int(cr.get("applied_candidate_id",0)) != cid:
            stats["candidate_unavailable"] += 1
            continue

        s0, margin0 = _score(teacher_res)
        s1, margin1 = _score(cand_res)
        ds = float(s1 - s0)
        dm = float(margin1 - margin0)

        if ds > 0:
            label, weight = 1, 12.0 + min(6.0, abs(dm)/5000.0)
            stats["wl_improve"] += 1
        elif ds < 0:
            label, weight = 0, 12.0 + min(6.0, abs(dm)/5000.0)
            stats["wl_harm"] += 1
        elif dm >= MARGIN_EPS:
            label, weight = 1, 1.0 + min(3.0, abs(dm)/10000.0)
            stats["margin_improve"] += 1
        elif dm <= -MARGIN_EPS:
            label, weight = 0, 1.0 + min(3.0, abs(dm)/10000.0)
            stats["margin_harm"] += 1
        else:
            stats["ambiguous"] += 1
            continue

        features = cr.get("features") or m["row"].get("features")
        if features is None:
            stats["missing_features"] += 1
            continue

        world = m["world"]
        samples.append({
            "features": [float(x) for x in features],
            "label": int(label),
            "weight": float(weight),
            "delta_score": ds,
            "delta_margin": dm,
            "seed": int(world["seed"]),
            "opponent": str(world["opponent"]),
            "seat": int(world["seat"]),
            "step": int(m["row"]["step"]),
            "candidate_id": cid,
            "round": int(round_id),
            "state_hash": str(cr.get("state_hash")),
        })
        stats["accepted"] += 1
        stats[f"cand_{cid}_n"] += 1
        stats[f"cand_{cid}_positive"] += int(label)

    print("round stats:", dict(stats))
    return samples, dict(stats), worlds

print("counterfactual collector ready")


In [ ]:

def evaluate_policy(policy_path, seeds, opponents=None):
    opponents = opponents or population_paths
    jobs, keys = [], []
    for opp_name, opp_path in opponents.items():
        for seed in seeds:
            for seat in (0,1):
                jobs.append({
                    "candidate_path": policy_path,
                    "opponent_path": opp_path,
                    "seed": int(seed),
                    "seat": int(seat),
                })
                keys.append((str(opp_name), int(seed), int(seat)))
    results = parallel_run(jobs)
    rows = []
    for key, result in zip(keys, results):
        opp, seed, seat = key
        sc, margin = _score(result)
        rows.append({
            "opponent": opp,
            "seed": seed,
            "seat": seat,
            "ok": bool(result and result.get("ok")),
            "score": None if sc is None else float(sc),
            "margin": None if margin is None else float(margin),
            "candidate_reward": None if not result else result.get("candidate_reward"),
            "opponent_reward": None if not result else result.get("opponent_reward"),
            "error": None if not result else result.get("error"),
        })
    return rows

def summarize_rows(rows):
    good = [r for r in rows if r["ok"] and r["score"] is not None]
    return {
        "games": len(rows),
        "ok": len(good),
        "failures": len(rows)-len(good),
        "win_score": float(np.mean([r["score"] for r in good])) if good else 0.0,
        "mean_margin": float(np.mean([r["margin"] for r in good])) if good else 0.0,
        "wins": sum(r["score"] == 1.0 for r in good),
        "ties": sum(r["score"] == 0.5 for r in good),
        "losses": sum(r["score"] == 0.0 for r in good),
    }

def paired_report(student_rows, teacher_rows, bootstrap=6000):
    smap = {(r["opponent"],r["seed"],r["seat"]):r for r in student_rows}
    tmap = {(r["opponent"],r["seed"],r["seat"]):r for r in teacher_rows}
    pair = []
    for k in sorted(set(smap) & set(tmap)):
        s,t = smap[k],tmap[k]
        if not s["ok"] or not t["ok"]:
            continue
        pair.append({
            "opponent":k[0], "seed":k[1], "seat":k[2],
            "delta_score":float(s["score"]-t["score"]),
            "delta_margin":float(s["margin"]-t["margin"]),
        })

    # Aggregate both seats before bootstrap: one opponent/seed is one random world.
    grouped = defaultdict(list)
    for r in pair:
        grouped[(r["opponent"],r["seed"])].append(r)
    worlds = []
    for (opp,seed), vals in grouped.items():
        worlds.append({
            "opponent":opp, "seed":seed,
            "delta_score":float(np.mean([v["delta_score"] for v in vals])),
            "delta_margin":float(np.mean([v["delta_margin"] for v in vals])),
        })

    ds = np.asarray([w["delta_score"] for w in worlds], dtype=float)
    dm = np.asarray([w["delta_margin"] for w in worlds], dtype=float)
    rng = np.random.default_rng(20260824)
    if len(ds):
        boot = np.asarray([
            float(np.mean(ds[rng.integers(0,len(ds),len(ds))]))
            for _ in range(bootstrap)
        ])
        ci = np.quantile(boot,[0.025,0.975]).tolist()
    else:
        ci = [0.0,0.0]

    by_opp = {}
    for opp in sorted({w["opponent"] for w in worlds}):
        vals = [w for w in worlds if w["opponent"] == opp]
        by_opp[opp] = {
            "n_worlds":len(vals),
            "mean_delta_score":float(np.mean([v["delta_score"] for v in vals])),
            "mean_delta_margin":float(np.mean([v["delta_margin"] for v in vals])),
        }
    return {
        "paired_games":len(pair),
        "worlds":len(worlds),
        "mean_delta_score":float(np.mean(ds)) if len(ds) else 0.0,
        "mean_delta_margin":float(np.mean(dm)) if len(dm) else 0.0,
        "score_ci95":ci,
        "by_opponent":by_opp,
    }

# Proof that a disabled residual is behaviorally identical to the frozen teacher
# in complete games before we allow any training.
_EQ_OPP = {
    "moon": population_paths["moon"],
    "munib_base": population_paths["munib_base"],
}
_EQ_SEEDS = [460001,460002,460003,460004]
_eq_teacher = evaluate_policy(str(TEACHER_PATH), _EQ_SEEDS, _EQ_OPP)
_eq_probe = evaluate_policy(PROBE_PATH, _EQ_SEEDS, _EQ_OPP)
_eq_t = {(r["opponent"],r["seed"],r["seat"]):r for r in _eq_teacher}
_eq_p = {(r["opponent"],r["seed"],r["seat"]):r for r in _eq_probe}
for k in _eq_t:
    a,b = _eq_t[k],_eq_p[k]
    assert a["ok"] and b["ok"], (k,a,b)
    assert a["candidate_reward"] == b["candidate_reward"], (k,a,b)
    assert a["opponent_reward"] == b["opponent_reward"], (k,a,b)
print("disabled residual == exact teacher on", len(_eq_t), "full games")


In [ ]:

from sklearn.ensemble import RandomForestClassifier

def _arrays(samples):
    if not samples:
        return None, None, None
    X = np.asarray([s["features"] for s in samples], dtype=np.float64)
    y = np.asarray([s["label"] for s in samples], dtype=np.int64)
    w = np.asarray([s["weight"] for s in samples], dtype=np.float64)
    assert X.ndim == 2
    assert len(X) == len(y) == len(w)
    return X, y, w

def export_forest(clf):
    trees = []
    classes = list(map(int, clf.classes_.tolist()))
    pos_index = classes.index(1)
    for est in clf.estimators_:
        t = est.tree_
        values = t.value[:, 0, :]
        probs = []
        for row in values:
            total = float(np.sum(row))
            probs.append(float(row[pos_index] / total) if total > 0 else 0.5)
        trees.append({
            "left": t.children_left.astype(int).tolist(),
            "right": t.children_right.astype(int).tolist(),
            "feature": t.feature.astype(int).tolist(),
            "threshold": t.threshold.astype(float).tolist(),
            "prob": probs,
        })
    return trees

def fit_residual_forest(samples, seed=12345):
    X, y, w = _arrays(samples)
    disabled = {
        "enabled": False,
        "trees": [],
        "threshold": 1.01,
        "max_std": 0.0,
        "thresholds": {str(i):1.01 for i in range(1,9)},
        "max_stds": {str(i):0.0 for i in range(1,9)},
        "min_priority": 0.0,
        "max_overrides": MAX_OVERRIDES,
    }
    if X is None or len(samples) < 80:
        print("model disabled: too few accepted counterfactuals", 0 if X is None else len(samples))
        return disabled, None
    classes, counts = np.unique(y, return_counts=True)
    if len(classes) < 2 or int(counts.min()) < 20:
        print("model disabled: insufficient class coverage", dict(zip(classes.tolist(), counts.tolist())))
        return disabled, None

    clf = RandomForestClassifier(
        n_estimators=N_TREES,
        max_depth=TREE_DEPTH,
        min_samples_leaf=MIN_LEAF,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=seed,
    )
    clf.fit(X, y, sample_weight=w)
    bundle = {
        "enabled": True,
        "trees": export_forest(clf),
        # Conservative defaults; replaced by disjoint calibration.
        "threshold": 0.90,
        "max_std": 0.28,
        "thresholds": {str(i):0.90 for i in range(1,9)},
        "max_stds": {str(i):0.28 for i in range(1,9)},
        "min_priority": 0.0,
        "max_overrides": MAX_OVERRIDES,
        "feature_dim": int(X.shape[1]),
        "train_rows": int(len(X)),
        "positive_rate": float(np.mean(y)),
    }
    return bundle, clf

def _tree_probs_from_bundle(bundle, x):
    ps = []
    for tree in bundle.get("trees", []) or []:
        left, right = tree["left"], tree["right"]
        feature, threshold, prob = tree["feature"], tree["threshold"], tree["prob"]
        node = 0
        while left[node] != -1:
            node = left[node] if float(x[feature[node]]) <= threshold[node] else right[node]
        ps.append(float(prob[node]))
    return ps

def bundle_predict(bundle, x):
    ps = _tree_probs_from_bundle(bundle, x)
    if not ps:
        return None, None
    p = float(np.mean(ps))
    std = float(np.std(ps))
    return p, std

def _cal_metrics(rows, bundle, threshold, max_std, candidate_id=None):
    grouped = defaultdict(list)
    for r in rows:
        if candidate_id is not None and int(r["candidate_id"]) != int(candidate_id):
            continue
        p, std = bundle_predict(bundle, r["features"])
        rr = dict(r)
        rr["p"] = p
        rr["std"] = std
        grouped[(r["seed"],r["opponent"],r["seat"],r["step"])].append(rr)

    chosen = []
    for _, vals in grouped.items():
        elig = [
            r for r in vals
            if r["p"] is not None and r["std"] is not None
            and r["p"] >= threshold and r["std"] <= max_std
        ]
        if not elig:
            continue
        best = max(elig, key=lambda r:(r["p"], r["delta_margin"]))
        chosen.append(best)

    if not chosen:
        return {
            "overrides":0, "mean_delta_score":0.0, "mean_delta_margin":0.0,
            "harmful_rate":0.0, "positive_rate":0.0,
        }
    return {
        "overrides": len(chosen),
        "mean_delta_score": float(np.mean([r["delta_score"] for r in chosen])),
        "mean_delta_margin": float(np.mean([r["delta_margin"] for r in chosen])),
        "harmful_rate": float(np.mean([
            (r["delta_score"] < 0) or (r["delta_score"] == 0 and r["delta_margin"] < -MARGIN_EPS)
            for r in chosen
        ])),
        "positive_rate": float(np.mean([r["label"] for r in chosen])),
    }

def calibrate_bundle(bundle, cal_samples):
    out = copy.deepcopy(bundle)
    if not out.get("enabled"):
        return out, {"enabled":False}

    thresholds = [0.60,0.65,0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.94,0.96]
    stds = [0.12,0.16,0.20,0.24,0.28,0.32,0.40]
    report = {}

    # Candidate-specific gates. A source with weak calibration evidence is disabled,
    # while other residual sources remain usable.
    for cid in range(1,9):
        rows = [r for r in cal_samples if int(r["candidate_id"]) == cid]
        best = None
        for th in thresholds:
            for st in stds:
                m = _cal_metrics(rows, out, th, st, candidate_id=cid)
                if m["overrides"] < 5:
                    continue
                # W/L/T dominates. Margin only separates equal ladder outcomes.
                objective = (
                    1000.0 * m["mean_delta_score"]
                    + m["mean_delta_margin"] / 5000.0
                    - 2.0 * m["harmful_rate"]
                    + 0.002 * min(m["overrides"], 50)
                )
                safe = (
                    m["mean_delta_score"] >= -1e-12
                    and m["mean_delta_margin"] >= 0.0
                    and m["harmful_rate"] <= 0.20
                )
                item = (safe, objective, th, st, m)
                if best is None or item[:2] > best[:2]:
                    best = item

        if best is None or not best[0]:
            out["thresholds"][str(cid)] = 1.01
            out["max_stds"][str(cid)] = 0.0
            report[str(cid)] = {
                "enabled":False,
                "rows":len(rows),
                "reason":"no safe calibrated operating point",
            }
        else:
            _, objective, th, st, m = best
            out["thresholds"][str(cid)] = float(th)
            out["max_stds"][str(cid)] = float(st)
            report[str(cid)] = {
                "enabled":True,
                "rows":len(rows),
                "threshold":float(th),
                "max_std":float(st),
                "objective":float(objective),
                **m,
            }

    enabled = [
        int(cid) for cid,v in report.items()
        if isinstance(v,dict) and v.get("enabled")
    ]
    out["enabled"] = bool(enabled)
    out["enabled_candidates"] = enabled

    # Global defaults are irrelevant when per-candidate keys exist, but make the
    # artifact self-describing.
    out["threshold"] = 0.90
    out["max_std"] = 0.28

    # Sanity report of the grouped calibrated policy on counterfactual calibration rows.
    grouped_policy = []
    groups = defaultdict(list)
    for r in cal_samples:
        p,std = bundle_predict(out, r["features"])
        rr = dict(r); rr["p"]=p; rr["std"]=std
        groups[(r["seed"],r["opponent"],r["seat"],r["step"])].append(rr)
    for _, vals in groups.items():
        elig = []
        for r in vals:
            cid = int(r["candidate_id"])
            th = float(out["thresholds"].get(str(cid),1.01))
            st = float(out["max_stds"].get(str(cid),0.0))
            if r["p"] is not None and r["std"] is not None and r["p"] >= th and r["std"] <= st:
                elig.append(r)
        if elig:
            grouped_policy.append(max(elig,key=lambda r:(r["p"],r["delta_margin"])))

    def grouped_policy_metrics(active_candidates):
        chosen = []
        groups = defaultdict(list)
        for r in cal_samples:
            cid = int(r["candidate_id"])
            if cid not in active_candidates:
                continue
            p,std = bundle_predict(out, r["features"])
            rr = dict(r); rr["p"]=p; rr["std"]=std
            groups[(r["seed"],r["opponent"],r["seat"],r["step"])].append(rr)
        for _, vals in groups.items():
            elig = []
            for r in vals:
                cid = int(r["candidate_id"])
                th = float(out["thresholds"].get(str(cid),1.01))
                st = float(out["max_stds"].get(str(cid),0.0))
                if r["p"] is not None and r["std"] is not None and r["p"] >= th and r["std"] <= st:
                    elig.append(r)
            if elig:
                chosen.append(max(elig,key=lambda r:(r["p"],r["delta_margin"])))
        metrics = {
            "enabled_candidates": sorted(active_candidates),
            "overrides": len(chosen),
            "mean_delta_score": float(np.mean([r["delta_score"] for r in chosen])) if chosen else 0.0,
            "mean_delta_margin": float(np.mean([r["delta_margin"] for r in chosen])) if chosen else 0.0,
            "harmful_rate": float(np.mean([
                (r["delta_score"] < 0) or (r["delta_score"] == 0 and r["delta_margin"] < -MARGIN_EPS)
                for r in chosen
            ])) if chosen else 0.0,
        }
        metrics["objective"] = (
            1000.0*metrics["mean_delta_score"]
            + metrics["mean_delta_margin"]/5000.0
            - 2.0*metrics["harmful_rate"]
            + 0.002*min(metrics["overrides"],50)
        )
        return metrics

    # Joint candidate selection matters because runtime picks the best eligible
    # residual at a state. Greedily prune sources until the combined policy is safe.
    active = set(enabled)
    joint = grouped_policy_metrics(active)
    while active and (
        joint["overrides"] < 5
        or joint["mean_delta_score"] < -1e-12
        or joint["mean_delta_margin"] < 0.0
        or joint["harmful_rate"] > 0.20
    ):
        trials = []
        for cid in sorted(active):
            m = grouped_policy_metrics(active - {cid})
            trials.append((m["objective"], cid, m))
        if not trials:
            break
        _, remove_cid, best_joint = max(trials, key=lambda z:z[0])
        active.remove(remove_cid)
        out["thresholds"][str(remove_cid)] = 1.01
        out["max_stds"][str(remove_cid)] = 0.0
        report[str(remove_cid)]["joint_pruned"] = True
        joint = best_joint

    out["enabled_candidates"] = sorted(active)
    out["enabled"] = bool(active and joint["overrides"] >= 5)
    if not out["enabled"]:
        for cid in range(1,9):
            out["thresholds"][str(cid)] = 1.01
            out["max_stds"][str(cid)] = 0.0
        out["enabled_candidates"] = []
        joint = grouped_policy_metrics(set())

    report["policy"] = joint
    return out, report

print("forest trainer + calibration ready")



## Training

Round 0 branches from states visited by the exact teacher. DAgger rounds branch from
states visited by the current learned residual. Every target decision is re-checked with
a SHA-256 hash of the complete observable state before accepting a label.

The forest is intentionally an abstaining model. Candidate-specific thresholds are set
only on disjoint calibration worlds and unsafe candidate families are pruned jointly.


In [ ]:

# Round 0: exact V21-R1 teacher distribution. This is the behavior-cloning anchor.
R0_SAMPLES, R0_STATS, R0_WORLDS = collect_counterfactual_round(
    PROBE_PATH,
    DISCOVERY_SEEDS,
    round_id=0,
    branch_budget=BRANCHES_R0,
)
assert R0_SAMPLES, "No valid teacher counterfactual samples were collected."
FEATURE_DIM = len(R0_SAMPLES[0]["features"])
assert all(len(s["features"]) == FEATURE_DIM for s in R0_SAMPLES)

BUNDLE_R0, CLF_R0 = fit_residual_forest(R0_SAMPLES, seed=1001)
R0_POLICY_PATH, _ = write_policy(BUNDLE_R0, "student_r0_probe.py", enable_log=True)

(WORK / "residual_r0_samples.json").write_text(json.dumps(R0_SAMPLES), encoding="utf-8")
(WORK / "residual_r0_model.json").write_text(json.dumps(BUNDLE_R0), encoding="utf-8")
print({
    "r0_samples": len(R0_SAMPLES),
    "feature_dim": FEATURE_DIM,
    "positive_rate": float(np.mean([s["label"] for s in R0_SAMPLES])),
    "student_enabled": bool(BUNDLE_R0.get("enabled")),
})


In [ ]:

# DAgger 1: reach states induced by the learned residual, then re-query the exact
# counterfactual teacher-vs-candidate oracle on those states.
R1_SAMPLES, R1_STATS, R1_WORLDS = collect_counterfactual_round(
    R0_POLICY_PATH,
    DAGGER1_SEEDS,
    round_id=1,
    branch_budget=BRANCHES_R1,
)
AGG_R1 = R0_SAMPLES + R1_SAMPLES
BUNDLE_R1, CLF_R1 = fit_residual_forest(AGG_R1, seed=2002)
R1_POLICY_PATH, _ = write_policy(BUNDLE_R1, "student_r1_probe.py", enable_log=True)

(WORK / "residual_r1_samples.json").write_text(json.dumps(R1_SAMPLES), encoding="utf-8")
(WORK / "residual_r1_model.json").write_text(json.dumps(BUNDLE_R1), encoding="utf-8")
print({
    "dagger1_new": len(R1_SAMPLES),
    "aggregate": len(AGG_R1),
    "student_enabled": bool(BUNDLE_R1.get("enabled")),
})


In [ ]:

# DAgger 2: one more state-distribution correction. Still no calibration leakage.
R2_SAMPLES, R2_STATS, R2_WORLDS = collect_counterfactual_round(
    R1_POLICY_PATH,
    DAGGER2_SEEDS,
    round_id=2,
    branch_budget=BRANCHES_R2,
)
TRAIN_SAMPLES = AGG_R1 + R2_SAMPLES
BUNDLE_PRECAL, CLF_PRECAL = fit_residual_forest(TRAIN_SAMPLES, seed=3003)
PRECAL_POLICY_PATH, _ = write_policy(BUNDLE_PRECAL, "student_precal_probe.py", enable_log=True)

(WORK / "residual_r2_samples.json").write_text(json.dumps(R2_SAMPLES), encoding="utf-8")
(WORK / "residual_train_samples.json").write_text(json.dumps(TRAIN_SAMPLES), encoding="utf-8")
(WORK / "residual_model_precal.json").write_text(json.dumps(BUNDLE_PRECAL), encoding="utf-8")
print({
    "dagger2_new": len(R2_SAMPLES),
    "train_total": len(TRAIN_SAMPLES),
    "student_enabled": bool(BUNDLE_PRECAL.get("enabled")),
})


In [ ]:

# Completely disjoint calibration worlds. These rows only set abstention thresholds;
# they are never added to the forest fit.
CAL_SAMPLES, CAL_STATS, CAL_WORLDS = collect_counterfactual_round(
    PRECAL_POLICY_PATH,
    CAL_SEEDS,
    round_id=3,
    branch_budget=BRANCHES_CAL,
)
BUNDLE_FINAL, CAL_REPORT = calibrate_bundle(BUNDLE_PRECAL, CAL_SAMPLES)

FINAL_STUDENT_PATH, FINAL_STUDENT_SOURCE = write_policy(
    BUNDLE_FINAL,
    "student_final.py",
    enable_log=False,
)
FINAL_STUDENT_LOG_PATH, _ = write_policy(
    BUNDLE_FINAL,
    "student_final_probe.py",
    enable_log=True,
)

assert FINAL_STUDENT_SOURCE.startswith(TEACHER_SOURCE.rstrip())
assert hashlib.sha256(TEACHER_SOURCE.encode()).hexdigest() == TEACHER_SHA

(WORK / "residual_cal_samples.json").write_text(json.dumps(CAL_SAMPLES), encoding="utf-8")
(WORK / "residual_model_final.json").write_text(json.dumps(BUNDLE_FINAL), encoding="utf-8")
(WORK / "residual_calibration_report.json").write_text(
    json.dumps(CAL_REPORT, indent=2), encoding="utf-8"
)
print(json.dumps(CAL_REPORT, indent=2))
print("enabled residual candidates:", BUNDLE_FINAL.get("enabled_candidates", []))



## Untouched promotion

The final student and the exact teacher face the same unseen opponents, seeds and both
seats. Promotion requires non-regression in paired W/L/T, non-negative paired margin,
a bootstrap lower bound, no technical failures, and a direct student-vs-teacher gate.

If those checks are not met, the notebook deliberately submits the original 2300 teacher.


In [ ]:

# Untouched full-game promotion gate. Neither forest fitting nor threshold calibration
# has seen these seeds.
print("HOLDOUT: student")
STUDENT_HOLDOUT = evaluate_policy(FINAL_STUDENT_PATH, HOLDOUT_SEEDS, population_paths)
print("HOLDOUT: exact teacher")
TEACHER_HOLDOUT = evaluate_policy(str(TEACHER_PATH), HOLDOUT_SEEDS, population_paths)

STUDENT_SUMMARY = summarize_rows(STUDENT_HOLDOUT)
TEACHER_SUMMARY = summarize_rows(TEACHER_HOLDOUT)
PAIRED = paired_report(STUDENT_HOLDOUT, TEACHER_HOLDOUT)

print("DIRECT student vs teacher")
DIRECT_ROWS = evaluate_policy(
    FINAL_STUDENT_PATH,
    DIRECT_SEEDS,
    {"teacher_v21_r1": str(TEACHER_PATH)},
)
DIRECT_SUMMARY = summarize_rows(DIRECT_ROWS)

enabled_candidates = list(BUNDLE_FINAL.get("enabled_candidates", []) or [])
worst_opp_delta = min(
    [v["mean_delta_score"] for v in PAIRED["by_opponent"].values()],
    default=0.0,
)

PROMOTION = bool(
    enabled_candidates
    and STUDENT_SUMMARY["failures"] == 0
    and TEACHER_SUMMARY["failures"] == 0
    and DIRECT_SUMMARY["failures"] == 0
    and STUDENT_SUMMARY["win_score"] >= TEACHER_SUMMARY["win_score"]
    and PAIRED["mean_delta_score"] >= 0.003
    and PAIRED["score_ci95"][0] >= -0.010
    and PAIRED["mean_delta_margin"] >= 0.0
    and DIRECT_SUMMARY["win_score"] >= 0.50
    and worst_opp_delta >= -0.04
)

PROMOTION_REPORT = {
    "promoted": PROMOTION,
    "enabled_candidates": enabled_candidates,
    "student": STUDENT_SUMMARY,
    "teacher": TEACHER_SUMMARY,
    "paired": PAIRED,
    "direct_vs_teacher": DIRECT_SUMMARY,
    "worst_opponent_delta": worst_opp_delta,
    "calibration": CAL_REPORT,
    "teacher_sha256": TEACHER_SHA,
}
(WORK / "residual_promotion_report.json").write_text(
    json.dumps(PROMOTION_REPORT, indent=2), encoding="utf-8"
)
print(json.dumps(PROMOTION_REPORT, indent=2))


In [ ]:

def package_source(source, tar_path, main_path=None):
    import io as _io, gzip as _gzip, tarfile as _tarfile
    if main_path is not None:
        Path(main_path).write_text(source, encoding="utf-8")
    raw_tar = _io.BytesIO()
    with _tarfile.open(fileobj=raw_tar, mode="w") as tar:
        payload = source.encode("utf-8")
        info = _tarfile.TarInfo("main.py")
        info.size = len(payload)
        info.mode = 0o644
        info.mtime = 0
        info.uid = info.gid = 0
        info.uname = info.gname = ""
        tar.addfile(info, _io.BytesIO(payload))
    with Path(tar_path).open("wb") as raw:
        with _gzip.GzipFile(filename="", mode="wb", fileobj=raw, mtime=0) as gz:
            gz.write(raw_tar.getvalue())
    with _tarfile.open(tar_path, "r:gz") as tar:
        assert tar.getnames() == ["main.py"]
        packed = tar.extractfile("main.py").read().decode("utf-8")
    assert packed == source
    return hashlib.sha256(Path(tar_path).read_bytes()).hexdigest()

# Always emit both. The selected submission is student only after untouched promotion.
STUDENT_ARCHIVE_SHA = package_source(
    FINAL_STUDENT_SOURCE,
    WORK / "submission_student_candidate.tar.gz",
    WORK / "main_student_candidate.py",
)
TEACHER_REPACK_SHA = package_source(
    TEACHER_SOURCE,
    WORK / "submission_teacher_repacked.tar.gz",
    WORK / "main_teacher_exact.py",
)

SELECTED_SOURCE = FINAL_STUDENT_SOURCE if PROMOTION else TEACHER_SOURCE
SELECTED_NAME = "teacher_anchored_residual_student" if PROMOTION else "exact_2300_teacher_fallback"
SELECTED_ARCHIVE_SHA = package_source(
    SELECTED_SOURCE,
    WORK / "submission.tar.gz",
    WORK / "main.py",
)
(WORK / "submission.py").write_text(SELECTED_SOURCE, encoding="utf-8")

FINAL_METRICS = {
    "selected": SELECTED_NAME,
    "promotion": PROMOTION_REPORT,
    "teacher_main_sha256": TEACHER_SHA,
    "student_main_sha256": hashlib.sha256(FINAL_STUDENT_SOURCE.encode()).hexdigest(),
    "selected_main_sha256": hashlib.sha256(SELECTED_SOURCE.encode()).hexdigest(),
    "student_archive_sha256": STUDENT_ARCHIVE_SHA,
    "teacher_repacked_archive_sha256": TEACHER_REPACK_SHA,
    "selected_archive_sha256": SELECTED_ARCHIVE_SHA,
    "feature_dim": FEATURE_DIM,
    "train_samples": len(TRAIN_SAMPLES),
    "cal_samples": len(CAL_SAMPLES),
    "round_stats": {
        "r0": R0_STATS,
        "r1": R1_STATS,
        "r2": R2_STATS,
        "cal": CAL_STATS,
    },
}
(WORK / "teacher_residual_metrics.json").write_text(
    json.dumps(FINAL_METRICS, indent=2), encoding="utf-8"
)

print({
    "SELECTED": SELECTED_NAME,
    "submit": str(WORK / "submission.tar.gz"),
    "main.py": str(WORK / "main.py"),
    "student_candidate": str(WORK / "submission_student_candidate.tar.gz"),
    "teacher_exact": str(WORK / "submission_teacher_exact.tar.gz"),
    "metrics": str(WORK / "teacher_residual_metrics.json"),
    "selected_sha256": hashlib.sha256(SELECTED_SOURCE.encode()).hexdigest(),
})



## Outputs

After the notebook finishes, `/kaggle/working` contains:

- `submission.tar.gz` — the **selected** artifact; learned residual only if promotion passed,
  otherwise the exact 2300 V21-R1 teacher.
- `main.py` — selected executable source.
- `submission_student_candidate.tar.gz` — learned candidate even if rejected.
- `submission_teacher_exact.tar.gz` — untouched original archive saved before training.
- `teacher_residual_metrics.json` — round statistics, calibration and paired holdout report.
- `residual_promotion_report.json` — the final promotion decision.

Submit `submission.tar.gz`.
